# Custom Standardizer

🧑🏻‍🏫 In this challenge, we will build our *own* StandardScaler. Why? Good question!

🎯 The goals of this exercise are:
- Understand the difference between `stateless transformers` and `stateful transformers`
- Learn how to work with `FeatureUnion`

## (1) 📚 Stateless Transformer vs. Stateful Transformer

🔢 Consider the following training set and test set...

In [1]:
import numpy as np
import pandas as pd

X_train = pd.DataFrame({
    'A': {0: 1, 1: 2, 2: 3},
    'B': {0: 4, 1: 5, 2: 6},
    'C': {0: 7, 1: 8, 2: 9}
})

print("This is the training dataset:")
display(X_train)

X_test = pd.DataFrame({
    'A': {0: 1, 1: 2, 2: 3},
    'B': {0: 2, 1: 3, 2: 4},
    'C': {0: 3, 1: 4, 2: 10}
})

print("This is the test dataset:")
display(X_test)

This is the training dataset:


,A,B,C
0,1,4,7
1,2,5,8
2,3,6,9


This is the test dataset:


,A,B,C
0,1,2,3
1,2,3,4
2,3,4,10


🛠 ...and consider the following FeatureUnion, which:
- Scales the features
- Creates a new feature that is the average of the other (unscaled) features

In [2]:
from sklearn import set_config; set_config(display='diagram')
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.pipeline import make_pipeline, make_union

standard_scaler = StandardScaler()
feature_averager = FunctionTransformer(lambda df: pd.DataFrame(1/3 * (df["A"] + df["B"] + df["C"])))

pipeline = make_union(standard_scaler, feature_averager)
pipeline

FeatureUnion(transformer_list=[('standardscaler', StandardScaler()),
                               ('functiontransformer',
                                FunctionTransformer(func=<function <lambda> at 0x126504c20>))])

Let's:
- Fit the pipeline on the training set
- Transform both the training set and the test set

In [3]:
pipeline.fit(X_train)

FeatureUnion(transformer_list=[('standardscaler', StandardScaler()),
                               ('functiontransformer',
                                FunctionTransformer(func=<function <lambda> at 0x126504c20>))])

In [4]:
X_train_transformed = pd.DataFrame(pipeline.transform(X_train))
X_train_transformed

,0,1,2,3
0,-1.224745,-1.224745,-1.224745,4.0
1,0.000000,0.000000,0.000000,5.0
2,1.224745,1.224745,1.224745,6.0


In [5]:
X_test_transformed = pd.DataFrame(pipeline.transform(X_test))
X_test_transformed

,0,1,2,3
0,-1.224745,-3.674235,-6.123724,2.000000
1,0.000000,-2.449490,-4.898979,3.000000
2,1.224745,-1.224745,2.449490,5.666667


👨🏻‍🏫 Notice how fundamentally different `StandardScaler` and `FunctionTransformer` are! ❗️

When we fit the pipeline and transform both the training set and the test set:

* **`FunctionTransformer (feature_averager)`**:
    * Learned nothing during *.fit()*
    * Simply applied a **stateless transformation**: $ \large (X_1, X_2, X_3) \rightarrow \frac{(X_1 + X_2 + X_3)}{3}$

* **`StandardScaler`**:
    * Learned the values of $\mu_{\color{blue}{train}}$ and $\sigma_{\color{blue}{train}}$ during *.fit()*
    * Used these learned values to perform a **stateful transformation** on both the training set and the test set:
        * $ \large X_{\color{blue}{train-scaled}} =  \frac{X_{\color{blue}{train}} -\mu_{\color{blue}{train}}}{\sigma_{\color{blue}{train}}}$
        * $ \large X_{\color{red}{test-scaled}} =  \frac{X_{\color{red}{test}} -\mu_{\color{blue}{train}}}{\sigma_{\color{blue}{train}}}$

## (2) 💻 Build Your Own Stateful Transformer

🤔 What if we want to code our own **stateful custom transformer**?

💪 We can write our own class!

# Custom Standardizer

### (2.1) 💻 Custom Standardizer

❓ **Task: Coding Your Own Class** ❓

1. Write your own `CustomStandardizer` class.
    * It should behave **exactly like** Scikit-learn's `StandardScaler`. In other words, it should have:
        * A `.fit()` method that calculates ("learns") the values of $\mu_{\color{blue}{train}}$ and $\sigma_{\color{blue}{train}}$.
        * A `.transform()` method.

2. Fit this class on `X_train`.

3. Apply the transformation to both `X_train` and `X_test`.

4. Compare your `CustomStandardizer` with Scikit-learn's `StandardScaler` to make sure everything is implemented correctly!

In [11]:
#########################################
# 1 - Implement the CustomStandardizer Class #
#########################################

# TransformerMixin inheritance provides the fit_transform() method based on fit() and transform().
from sklearn.base import TransformerMixin, BaseEstimator



class CustomStandardizer(TransformerMixin, BaseEstimator):

    def __init__(self):
        pass

    def fit(self, X, y=None):
        '''
       Stores the required values as instance attributes.
       Returns "self" to allow chaining fit and transform.
        '''
        self.mean_ = X.mean()
        self.std_ = X.std(ddof=0)
        return self

    def transform(self, X, y=None):
        return (X - self.mean_) / self.std_

    def inverse_transform(self, X, y=None):
        return X * self.std_ + self.mean_


In [12]:
#########################################
# 2 - Fit the CustomStandardizer Class #
#########################################

custom_standardizer = CustomStandardizer()
custom_standardizer.fit(X_train)

CustomStandardizer()

In [13]:
#########################################
# 3 - Transform                         #
#########################################

X_train_transformed = custom_standardizer.transform(X_train)
X_test_transformed = custom_standardizer.transform(X_test)


🧪 **Test Your Code**

In [14]:
from nbresult import ChallengeResult

tmp = CustomStandardizer()
tmp_train = np.array(tmp.fit_transform(X_train))
tmp_test = np.array(tmp.transform(X_test))

result = ChallengeResult(
    'standardizer',
    X_train_transformed=tmp_train,
    X_test_transformed=tmp_test
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/laraolgun/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/laraolgun/code/laraolgun74-creator/S17D1-S-Data-hand-made-standardizer/tests
plugins: anyio-4.8.0, dash-4.4.1, typeguard-4.4.2
collecting ... collected 1 item

test_standardizer.py::TestStandardizer::test_solution PASSED             [100%]

============================== 1 passed in 0.16s ===============================


💯 You can commit your code:

git add tests/standardizer.pickle

git commit -m 'Completed standardizer step'

git push origin master



<details>
<summary>💡 <i>Hints</i> (🧪 if the tests above fail by only a small margin)</summary>

* Note that there is a **small difference** between `np.std()` and `pd.std()`!

* This [Stack Overflow post](https://stackoverflow.com/questions/44220290/sklearn-standardscaler-result-different-to-manual-result) may help 😉

</details>

<details>
<summary>💡 <i>Solution</i></summary>

If you use the `pd.std` method, you also need to use the `ddof=0` parameter to reproduce the exact behavior of scikit-learn's `StandardScaler`.

</details>

### (2.2) 💻 Inverse Transform

❓ **Question (Inverse Transform)** ❓

Scikit-learn's `StandardScaler` has an [`.inverse_transform()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html#sklearn.preprocessing.StandardScaler.inverse_transform) method that allows you to revert to the original, unscaled dataset.

1. Go back to your `CustomStandardizer` class and implement your own `.inverse_transform()` method.

2. Try it on your scaled training set and your scaled test set.

In [15]:
X_train_inverse_transformed = custom_standardizer.inverse_transform(X_train_transformed)
X_test_inverse_transformed = custom_standardizer.inverse_transform(X_test_transformed)

In [ ]:
# Try your inverse transformation and run the following test to make sure it is implemented correctly.

🧪 **Test Your Code**

In [16]:
assert np.allclose(X_train_inverse_transformed, X_train)
assert np.allclose(X_test_inverse_transformed, X_test)

### (2.3) 💻 Custom Pipeline Completed!

💪 You have successfully replicated Scikit-learn's `StandardScaler`!

🌶 Now let's make things a little more exciting!

❓ **Task: Enhance Your Previous `CustomStandardizer` Custom Transformer with a Shrink Factor** ❓

Your modified `CustomStandardizer(shrink_factor=1)` class should accept an **additional argument** to make the scaling more pronounced. In this case, the scaling will be **inversely proportional** to the shrink factor 👇:

- $ \large X_{\color{blue}{train-scaled}} =  \left(\frac{X_{\color{blue}{train}} - \mu_{\color{blue}{train}}}{\sigma_{\color{blue}{train}}}\right) \times \frac{1}{shrink\ factor}$
- $ \large X_{\color{red}{test-scaled}} =  \left(\frac{X_{\color{red}{test}} - \mu_{\color{blue}{train}}}{\sigma_{\color{blue}{train}}}\right) \times \frac{1}{shrink\ factor}$

In [20]:
###################################
# Custom Standardizer             #
###################################



class CustomStandardizer(TransformerMixin, BaseEstimator):

    def __init__(self, shrink_factor = 1):
        self.shrink_factor = shrink_factor

    def fit(self, X, y=None):
        '''
        Stores what needs to be stored as instance attributes.
        Returns "self" to allow chaining fit and transform.
        '''
        self.mean_ = X.mean()
        self.std_ = X.std(ddof=0)
        return self

    def transform(self, X, y=None):
        return (X - self.mean_) / self.std_ / self.shrink_factor

    def inverse_transform(self, X, y=None):
        return X * self.std_ * self.shrink_factor + self.mean_

🧪 **Test Your New `CustomStandardizer` Custom Transformer** (with `shrink_factor = 2`):

- Fit it on `X_train`.
- Apply the transformation to both `X_train` and `X_test`.
- Store the transformed DataFrames in variables named `X_train_transformed` and `X_test_transformed`.

# Custom Standardizer

In [22]:
tmp = CustomStandardizer(shrink_factor=2).fit(X_train)

X_train_transformed = tmp.transform(X_train)
X_test_transformed = tmp.transform(X_test)

🧪 **Test Your Code**

In [23]:
from nbresult import ChallengeResult

tmp = CustomStandardizer(shrink_factor=2).fit(X_train)
tmp_train = np.array(tmp.fit_transform(X_train))
tmp_test = np.array(tmp.transform(X_test))

result = ChallengeResult(
    'new_standardizer',
    X_train_transformed=tmp_train,
    X_test_transformed=tmp_test
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/laraolgun/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/laraolgun/code/laraolgun74-creator/S17D1-S-Data-hand-made-standardizer/tests
plugins: anyio-4.8.0, dash-4.4.1, typeguard-4.4.2
collecting ... collected 1 item

test_new_standardizer.py::TestNewStandardizer::test_solution PASSED      [100%]

============================== 1 passed in 0.15s ===============================


💯 You can commit your code:

git add tests/new_standardizer.pickle

git commit -m 'Completed new_standardizer step'

git push origin master



In [ ]:
# Run the following cells to make sure you obtain the correct transformations.
# truth_train = np.array([
#     [-0.612372, -0.612372, -0.612372],
#     [0.000000, 0.000000, 0.000000],
#     [0.612372, 0.612372, 0.612372]
# ])
# truth_test = np.array([
#     [-0.612372, -1.837117, -3.061862],
#     [ 0.        , -1.224745, -2.449490],
#     [ 0.612372, -0.612372,  1.224745]])

In [ ]:
# # Assertions
# np.allclose(X_train_transformed, truth_train)

In [ ]:
# # Test Assertions
# np.allclose(X_test_transformed, truth_test)

❓ **Task: Modify Your Previous `FeatureAverager` Custom Transformer** ❓

This modified `FeatureAverager()` class:
- Still calculates the average of three different features...
- ...and then divides the result by the maximum value in each row.

_Note_: Don't try to interpret this operation. It has no real meaning; our goal is simply to practice our custom class implementation skills.

$$(X_1, X_2, X_3) \rightarrow \frac{mean(X_1 + X_2 + X_3)}{max(X_1, X_2, X_3)}$$

_Note_: Think carefully about whether this is a **stateful** or **stateless** transformation (in other words, do we need to learn information about the entire training data before transforming, as we did with the stateful `StandardScaler`, or does it work like the *stateless* `feature_averager` above?). If it is the former, we need to implement something in `.fit()`; otherwise, we don't!

In [24]:
###################################
# Feature Averager                #
###################################

class FeatureAverager(TransformerMixin, BaseEstimator):

    def __init__(self):
        pass

    def fit(self, X, y=None):
        '''
        If necessary, this method stores information as instance attributes.
        Returns "self" to allow chaining fit and transform.
        '''
        return self

    def transform(self, X, y=None):
        return pd.DataFrame(X.mean(axis=1) / X.max(axis=1))

🧪 **Test Your `FeatureAverager` Custom Transformer** by fitting it on `X_train` and transforming both `X_train` and `X_test`.

In [25]:
feature_averager = FeatureAverager()
feature_averager.fit(X_train)

X_train_transformed = feature_averager.transform(X_train)
X_test_transformed = feature_averager.transform(X_test)

In [26]:
from nbresult import ChallengeResult

tmp = FeatureAverager()
tmp_train = np.array(tmp.fit_transform(X_train))
tmp_test = np.array(tmp.transform(X_test))

result = ChallengeResult(
    'feature_averager',
    X_train_transformed=tmp_train,
    X_test_transformed=tmp_test
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/laraolgun/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/laraolgun/code/laraolgun74-creator/S17D1-S-Data-hand-made-standardizer/tests
plugins: anyio-4.8.0, dash-4.4.1, typeguard-4.4.2
collecting ... collected 1 item

test_feature_averager.py::TestFeatureAverager::test_solution PASSED      [100%]

============================== 1 passed in 0.19s ===============================


💯 You can commit your code:

git add tests/feature_averager.pickle

git commit -m 'Completed feature_averager step'

git push origin master



❓ **Task (Feature Union)** ❓

1. Create a `FeatureUnion` pipeline using both `CustomStandardizer` and `FeatureAverager`, and store it in a variable named `pipeline`.

2. Fit the pipeline on `X_train` and transform both `X_train` and `X_test` (`shrink_factor = 3`).

3. Make sure to pass the final test of this challenge.

In [27]:
#####################
# 1 - Feature Union #
#####################

pipeline = make_union(
    CustomStandardizer(shrink_factor=3),
    FeatureAverager()
)

In [28]:
#########################
# 2 - Fit and Transform #
#########################

pipeline.fit(X_train)

X_train_transformed = pipeline.transform(X_train)
X_test_transformed = pipeline.transform(X_test)

In [29]:
from nbresult import ChallengeResult

tmp = pipeline
tmp_train = np.array(tmp.fit_transform(X_train))
tmp_test = np.array(tmp.transform(X_test))



result = ChallengeResult(
    'feature_union_custom_transformers',
    X_train_transformed=tmp_train,
    X_test_transformed=tmp_test
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/laraolgun/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/laraolgun/code/laraolgun74-creator/S17D1-S-Data-hand-made-standardizer/tests
plugins: anyio-4.8.0, dash-4.4.1, typeguard-4.4.2
collecting ... collected 1 item

test_feature_union_custom_transformers.py::TestFeatureUnionCustomTransformers::test_solution PASSED [100%]

============================== 1 passed in 0.17s ===============================


💯 You can commit your code:

git add tests/feature_union_custom_transformers.pickle

git commit -m 'Completed feature_union_custom_transformers step'

git push origin master



🏁 Congratulations! You have learned how to build your own custom transformers!

💾 Don't forget to add, commit, and push your notebook to GitHub...

🚀 ...and move on to the next challenge!